# NB08 · Capstone：Episode Quality Score v0 + 校准实验

| | |
|---|---|
| **目标** | 把前 7 本的全部能力拼成一个可校准的数据质量分，并用下游训练验证它——observatory 的种子、你的 signature 产出 |
| **前置** | NB01/03/04 全部完成（复用其结论与机器） |
| **预计耗时** | 2–3 天（含训练机器时） |
| **产出物** | `results/NB08.json`（EQS 权重 + 校准结果） |
| **通过标准** | 三组对照训练完成，能回答「EQS 排名靠前的数据是否真的训出更好的模型」 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nbutils

ds = nbutils.load_lerobot_dataset("lerobot/pusht")
N_EP = ds.num_episodes
print(f"episodes: {N_EP}")

In [ ]:
# 逐 episode 特征提取（EQS 的原材料）。跑一遍约几分钟。
feats = []
frame_idx = 0
for ep in range(N_EP):
    ep_frames = []
    while frame_idx < ds.num_frames:
        f = ds[frame_idx]
        if f["episode_index"].item() != ep:
            break
        ep_frames.append(f); frame_idx += 1
    A = np.stack([f["action"].numpy() for f in ep_frames])
    jerk = np.abs(np.diff(A, n=2, axis=0)).mean() if len(A) > 2 else 0.0
    feats.append({
        "episode": ep,
        "length": len(A),
        "smoothness": -jerk,                       # 越平滑越高
        "final_reward": float(ep_frames[-1].get("next.reward", ep_frames[-1].get("reward", 0.0)) or 0.0),
        "action_mean": A.mean(axis=0),
    })
print(f"extracted {len(feats)} episodes")

In [ ]:
# 多样性与重复度：与数据集质心的距离（多样性 proxy），近邻相似度（重复 proxy）
M = np.stack([f["action_mean"] for f in feats])
centroid = M.mean(axis=0)
for f in feats:
    f["diversity"] = float(np.linalg.norm(f["action_mean"] - centroid))
d = np.linalg.norm(M[:, None, :] - M[None, :, :], axis=-1)
np.fill_diagonal(d, np.inf)
for i, f in enumerate(feats):
    f["redundancy"] = -float(d[i].min())          # 最近邻越近 → 冗余越高 → 分越低

In [ ]:
# EQS v0：标准化后的加权和。权重是假设，下面的校准实验负责证伪它。
WEIGHTS = {"length": 0.1, "smoothness": 0.25, "final_reward": 0.25, "diversity": 0.25, "redundancy": 0.15}

def z(x):
    x = np.array(x, dtype=float); return (x - x.mean()) / (x.std() + 1e-9)

cols = {k: z([f[k] for f in feats]) for k in WEIGHTS}
eqs = sum(WEIGHTS[k] * cols[k] for k in WEIGHTS)
order = np.argsort(-eqs)
plt.figure(figsize=(7, 3)); plt.hist(eqs, bins=30); plt.title("EQS v0 distribution")
plt.savefig("results/NB08_eqs.png", dpi=120, bbox_inches="tight"); plt.show()
print("top-5 episodes:", order[:5].tolist(), " bottom-5:", order[-5:].tolist())
# ✅ 检查点：肉眼抽查——用 NB01 的轨迹可视化看 top-2 和 bottom-2 各长什么样，EQS 排的合理吗？

In [ ]:
# 校准实验：EQS 说了算不算？三组各 100 条 → 训练 → NB03 协议评测（复用 NB04 的训练矩阵机器）
K = 100
groups = {
    "top_eqs":  order[:K].tolist(),
    "random":   np.random.default_rng(0).choice(N_EP, K, replace=False).tolist(),
    "bottom_eqs": order[-K:].tolist(),
}
for name, eps in groups.items():
    print(f"{name}: 用 --dataset.episodes={sorted(eps)[:8]}... 训练（命令模板同 NB04）")
# 跑完把三个 success rate 填进来：
CALIBRATION = {"top_eqs": None, "random": None, "bottom_eqs": None}

In [ ]:
assert all(v is not None for v in CALIBRATION.values()), "三组训练+评测完成后填入"
n_eval = nbutils.latest("NB03")["n_episodes"]
plt.figure(figsize=(5, 3.5))
for i, (name, v) in enumerate(CALIBRATION.items()):
    lo, hi = nbutils.wilson_ci(int(v * n_eval), n_eval)
    plt.bar(i, v); plt.plot([i, i], [lo, hi], color="k")
plt.xticks(range(3), list(CALIBRATION)); plt.ylabel("success rate"); plt.title("EQS calibration")
plt.savefig("results/NB08_calibration.png", dpi=120, bbox_inches="tight"); plt.show()
nbutils.log_result("NB08", {"weights": WEIGHTS, "calibration": CALIBRATION, "k": K})

## 分析

1. **判决**：top > random > bottom 成立吗（按 NB03 的 CI 规则）？
   - 成立 → EQS v0 有信息量。哪个特征贡献最大？做 leave-one-out：去掉一个特征重排序，校准差多少。
   - top ≈ random → EQS 没测到真正的质量。是特征不对（该加什么？状态覆盖？接触事件？）还是"这个任务里数据质量本来就不是瓶颈"（对照 NB04 曲线的饱和段想想）？
   - **bottom ≈ top → 最有趣的结果**：说明这个数据集内部质量方差很小，EQS 该去更脏的数据集（DROID、社区采集）上证明自己。
2. **写成 spec**：把权重、标定方法、局限写成 `EQS_v0_spec.md`——这就是课表 Week 6 的公开交付物，也是 observatory/quality 的第一个模块。
3. **诚实条款**：无论结果如何都公开。负结果 +「为什么」的分析，比一个不可复现的正结果值钱十倍——这正是你与只发 demo 的人的区别。

## 毕业检查表

- [ ] NB00–NB08 全部有落盘结果（`results/` 下 9 个 json + 图）
- [ ] 每本的复盘都已填写并 commit
- [ ] NB03 的评测协议在 04/06/07/08 里被一致执行
- [ ] NB04 缩放曲线 + NB08 校准图 已准备好放进《数据资产审计》
- [ ] 下一步：把 profiler(NB01)、协议(NB03)、EQS(NB08) 迁入 physical-ai-data-observatory 独立仓库

到这里，你已经**单人闭环**了：数据解剖 → 训练 → 统计意义上可信的评测 → 缩放定价 → 失败归因 → 科学迭代 → 跨协议对比 → 质量分校准。这套账本就是你的第一份专家证据。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
